# P02 — Auditoría de fairness del modelo de Velora Finance

En P01 evaluamos el rendimiento **global** del modelo.

Ahora hacemos una pregunta diferente:

> ¿El sistema se comporta de forma comparable entre los grupos que queremos auditar?

`customer_group` no se utiliza como predictor. Se conserva únicamente para analizar el comportamiento del modelo por grupos.

El código está preparado. La interacción principal consiste en cambiar el **threshold** y observar qué criterios de fairness mejoran, empeoran o entran en conflicto.


## 1. Preparar el mismo caso de P01

Recreamos exactamente el dataset, la partición train/test y el modelo de P01 para que los resultados sean comparables.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", None)


In [ ]:
GENERATOR_URL = "https://raw.githubusercontent.com/pacosepulveda/ia-confiable-explicable-labs/main/resources/generate_velora_credit.py"

namespace = {"__name__": "velora_generator"}
with urllib.request.urlopen(GENERATOR_URL) as response:
    exec(response.read().decode("utf-8"), namespace)

df = namespace["generate_dataset"]()
print("Dataset cargado:", df.shape)


In [ ]:
FEATURES = [
    "age",
    "annual_income",
    "debt_ratio",
    "employment_months",
    "late_payments_12m",
    "credit_history_months",
    "existing_debt",
    "requested_amount",
    "savings_balance",
]

X = df[FEATURES]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

results = df.loc[X_test.index, ["application_id", "customer_group", "default"]].copy()
results["prob_default"] = proba

print("Modelo base recreado.")


## 2. Antes de mirar las predicciones: ¿los grupos parten de la misma situación?

Primero observamos tamaño y tasa real de impago por grupo.

Esto **no determina** por sí solo qué criterio de fairness debemos utilizar, pero sí evita interpretar las métricas sin contexto.


In [ ]:
group_context = (
    results.groupby("customer_group")
    .agg(
        solicitudes=("default", "size"),
        tasa_real_impago=("default", "mean"),
        probabilidad_media_predicha=("prob_default", "mean"),
    )
)

display(group_context.style.format({
    "tasa_real_impago": "{:.2%}",
    "probabilidad_media_predicha": "{:.2%}",
}))


### Cómo interpretar el resultado

Los grupos no tienen exactamente la misma tasa observada de impago.

Esto es importante porque varios criterios de fairness pueden producir conclusiones diferentes cuando las tasas base son distintas.

No debemos escoger una métrica únicamente porque produzca el resultado que esperamos.


## 3. Medir el comportamiento por grupo

Usaremos varias métricas simultáneamente.

Para evitar ambigüedad:

- **tasa de aprobación**: proporción clasificada como `no impago`;
- **detección de impago**: proporción de impagos reales correctamente detectados;
- **FPR**: clientes solventes marcados como riesgo;
- **FNR**: impagos reales que el sistema deja escapar;
- **precision**: de los marcados como riesgo, cuántos terminan realmente en impago.


In [ ]:
def metrics_by_group(data, threshold=0.50):
    temp = data.copy()
    temp["pred_default"] = (temp["prob_default"] >= threshold).astype(int)

    rows = []

    for group, part in temp.groupby("customer_group"):
        tn, fp, fn, tp = confusion_matrix(
            part["default"],
            part["pred_default"],
            labels=[0, 1]
        ).ravel()

        rows.append({
            "grupo": group,
            "n": len(part),
            "tasa_real_impago": part["default"].mean(),
            "tasa_alto_riesgo": part["pred_default"].mean(),
            "tasa_aprobacion": 1 - part["pred_default"].mean(),
            "accuracy": accuracy_score(part["default"], part["pred_default"]),
            "precision_riesgo": precision_score(
                part["default"], part["pred_default"], zero_division=0
            ),
            "deteccion_impago_TPR": tp / (tp + fn) if tp + fn else np.nan,
            "FPR": fp / (fp + tn) if fp + tn else np.nan,
            "FNR": fn / (fn + tp) if fn + tp else np.nan,
            "aprobacion_solventes": tn / (tn + fp) if tn + fp else np.nan,
            "aprobacion_impagos": fn / (fn + tp) if fn + tp else np.nan,
        })

    return pd.DataFrame(rows).set_index("grupo")


In [ ]:
threshold = 0.50
group_metrics = metrics_by_group(results, threshold)

display(group_metrics.style.format({
    c: "{:.2%}" for c in group_metrics.columns if c != "n"
}))


### Qué deberíamos observar

Con `threshold = 0.50` aparece un contraste relevante:

- las tasas globales de aprobación pueden parecer relativamente próximas;
- el **FPR** también puede ser parecido;
- pero la **detección de impagos** puede ser muy diferente entre grupos;
- en consecuencia, el **FNR** puede afectar mucho más a uno de ellos.

La pregunta deja de ser simplemente:

> ¿Tiene el modelo buena accuracy?

y pasa a ser:

> ¿Qué tipo de error estamos distribuyendo entre grupos?


## 4. Tres criterios pueden contar tres historias distintas

Vamos a calcular diagnósticos sencillos.

### Demographic parity

Para este caso definimos el resultado favorable como **aprobación**.  
Comparamos la tasa de aprobación entre grupos.

### Equal opportunity

Si consideramos como resultado favorable que un cliente solvente sea aprobado, comparamos:

`P(aprobado | realmente solvente)`

En este notebook definimos explícitamente **aprobación/solvencia como resultado favorable**. Si se cambia qué clase se considera positiva, también cambia la interpretación de estas métricas.

### Equalized odds

Además de lo anterior, exige controlar también cómo se comporta el sistema con los casos realmente insolventes.

No existe una regla que diga que una de estas métricas sea siempre la correcta para cualquier sistema.


In [ ]:
def fairness_diagnostics(group_table):
    approval_gap = (
        group_table["tasa_aprobacion"].max()
        - group_table["tasa_aprobacion"].min()
    )

    equal_opportunity_gap = (
        group_table["aprobacion_solventes"].max()
        - group_table["aprobacion_solventes"].min()
    )

    insolvent_approval_gap = (
        group_table["aprobacion_impagos"].max()
        - group_table["aprobacion_impagos"].min()
    )

    equalized_odds_gap = max(
        equal_opportunity_gap,
        insolvent_approval_gap
    )

    risk_detection_gap = (
        group_table["deteccion_impago_TPR"].max()
        - group_table["deteccion_impago_TPR"].min()
    )

    return pd.Series({
        "Demographic parity gap (aprobación)": approval_gap,
        "Equal opportunity gap (solventes)": equal_opportunity_gap,
        "Equalized odds gap": equalized_odds_gap,
        "Gap en detección de impagos": risk_detection_gap,
    })

diagnostics = fairness_diagnostics(group_metrics)
display(diagnostics.to_frame("diferencia").style.format("{:.2%}"))


### Interpretación

Es posible que un criterio muestre una diferencia pequeña y otro una diferencia considerable.

Eso **no es una contradicción del cálculo**.

Significa que cada criterio formaliza una idea diferente de qué significa un trato comparable.

Por eso una auditoría de fairness debe comenzar por identificar:

1. qué daño queremos detectar;
2. qué resultado consideramos favorable;
3. qué errores son especialmente relevantes;
4. qué grupos y subgrupos debemos analizar.


## 5. Cambiar el threshold también cambia la auditoría

El modelo sigue siendo exactamente el mismo.

Modifica el control y observa cómo cambian simultáneamente:

- rendimiento;
- aprobación;
- FPR y FNR;
- diferencias entre grupos.


In [ ]:
THRESHOLD_ELEGIDO = 0.50 #@param {type:"slider", min:0.10, max:0.80, step:0.05}

selected_metrics = metrics_by_group(results, THRESHOLD_ELEGIDO)
selected_diagnostics = fairness_diagnostics(selected_metrics)

print(f"Threshold: {THRESHOLD_ELEGIDO:.2f}")
print()

display(selected_metrics.style.format({
    c: "{:.2%}" for c in selected_metrics.columns if c != "n"
}))

print()
display(selected_diagnostics.to_frame("diferencia").style.format("{:.2%}"))


### Preguntas para discutir

- ¿Hay algún threshold en el que desaparezcan todas las diferencias?
- ¿Mejorar la paridad de aprobación implica necesariamente mejorar la detección de impagos?
- ¿Es aceptable optimizar una única métrica de fairness ignorando las demás?
- ¿Qué información adicional necesitaríamos antes de decidir que existe —o no existe— un problema de fairness?


## 6. “No usamos `customer_group`” no significa “no puede haber disparidad”

Comprobamos que `customer_group` no está dentro de las variables utilizadas por el modelo.


In [ ]:
print("Variables del modelo:")
for feature in FEATURES:
    print(" -", feature)

print()
print("¿customer_group es predictor?", "customer_group" in FEATURES)


### Idea clave

Un modelo puede producir comportamientos diferentes entre grupos aunque el atributo de grupo no se utilice directamente.

Las diferencias pueden aparecer por:

- otras variables correlacionadas;
- diferencias en las distribuciones de datos;
- variables omitidas;
- calidad de datos desigual;
- decisiones previas del proceso que generó los datos.

Eliminar un atributo sensible no constituye, por sí solo, una estrategia de fairness.


## 7. Profundización: calibración por grupo

Una probabilidad del 30 % debería tener un significado empírico razonablemente consistente.

Agruparemos las probabilidades del conjunto de test en intervalos comunes y compararemos:

- riesgo medio predicho;
- tasa real de impago observada.


In [ ]:
quantile_edges = np.unique(
    np.quantile(results["prob_default"], [0, .2, .4, .6, .8, 1.0])
)

cal = results.copy()
cal["bin"] = pd.cut(
    cal["prob_default"],
    bins=quantile_edges,
    include_lowest=True,
    duplicates="drop"
)

calibration_table = (
    cal.groupby(["customer_group", "bin"], observed=True)
       .agg(
           n=("default", "size"),
           riesgo_predicho=("prob_default", "mean"),
           impago_observado=("default", "mean"),
       )
       .reset_index()
)

display(calibration_table.style.format({
    "riesgo_predicho": "{:.2%}",
    "impago_observado": "{:.2%}",
}))


In [ ]:
for group in sorted(calibration_table["customer_group"].unique()):
    part = calibration_table[calibration_table["customer_group"] == group]
    plt.plot(
        part["riesgo_predicho"],
        part["impago_observado"],
        marker="o",
        label=f"Grupo {group}"
    )

plt.plot([0, 0.6], [0, 0.6], "--", label="Calibración ideal")
plt.xlabel("Riesgo medio predicho")
plt.ylabel("Impago observado")
plt.title("Calibración por grupo")
plt.legend()
plt.show()


### Cómo interpretar la calibración

Una curva próxima a la diagonal indica que las probabilidades predichas se corresponden razonablemente con las frecuencias observadas.

La calibración es otra dimensión distinta de la paridad de tasas o de errores.

Un sistema puede estar bien calibrado y, aun así, no satisfacer otros criterios de fairness.

De nuevo, no existe una única cifra que certifique que el sistema es “justo”.


# Conclusiones de P02

- Las métricas globales pueden ocultar comportamientos distintos entre grupos.
- No utilizar directamente el atributo de grupo **no garantiza** ausencia de disparidad.
- Demographic parity, equal opportunity, equalized odds y calibración responden a preguntas diferentes.
- Cambiar el threshold modifica tanto el rendimiento como las métricas de fairness.
- Una diferencia estadística es una señal para investigar; no constituye por sí sola una conclusión ética o legal.
- La elección de criterios debe relacionarse con el daño, el contexto y la decisión que el sistema soporta.

En P03 añadiremos otra dimensión de confiabilidad:

> ¿Qué ocurre cuando queremos obtener utilidad de los datos sin exponer más información de la necesaria?
